In [ ]:
"""
DS-505 Practical: Dataset Ingestion, Inspection and Descriptive Statistics
Simple student-friendly program based on the uploaded workbook.
"""

import pandas as pd
import numpy as np

# Step 1: Create a representative customer sales dataset
data = {
    'Transaction_ID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'Customer_Name': ['Aarav', 'Diya', 'Rohan', 'Ananya', 'Vivaan', 'Isha', 'Kabir', 'Meera', 'Aditya', 'Sanya'],
    'Age': [28, 34, 22, 45, 31, 29, 52, 26, 38, 41],
    'Department': ['Electronics', 'Clothing', 'Electronics', 'Home', 'Clothing', 'Electronics', 'Home', 'Clothing', 'Electronics', 'Home'],
    'Purchase_Amount': [12500.50, 4200.00, 18900.75, 3100.25, 6500.00, 14200.00, 8900.50, 5100.00, 22400.00, 7800.25],
    'Satisfaction_Score': [4.5, 3.8, 4.9, 3.2, 4.0, 4.7, 3.5, 4.2, 4.8, 3.9]
}

df = pd.DataFrame(data)

# Step 2: Inspect DataFrame structure
print("--- FIRST 5 RECORDS (head) ---")
print(df.head())

print("\n--- LAST 5 RECORDS (tail) ---")
print(df.tail())

print("\n--- DATASET SHAPE (Rows, Columns) ---")
print(df.shape)

print("\n--- COLUMN NAMES ---")
print(df.columns.tolist())

print("\n--- DATASET INFORMATION & DTYPES ---")
df.info()

print("\n--- DESCRIPTIVE SUMMARY STATISTICS ---")
print(df.describe(include='all'))


In [ ]:
#2
"""
DS-505 Practical: Data Hygiene, Missing Values, Deduplication and Renaming
Simple student-friendly program based on the uploaded workbook.
"""

import pandas as pd
import numpy as np

# Step 1: Create a dataset with missing values and duplicate records
raw_data = {
    'cust_id': [1, 2, 3, 4, 5, 2, 6, 7],
    'cust_name': ['Amit', 'Pooja', 'Rahul', 'Sneha', 'Vikram', 'Pooja', 'Karan', 'Neha'],
    'age': [25, 30, np.nan, 22, 28, 30, np.nan, 35],
    'monthly_income': [45000, 52000, 61000, np.nan, 48000, 52000, 55000, 70000]
}

df = pd.DataFrame(raw_data)
print("--- ORIGINAL DATASET ---")
print(df)

# Step 2: Check missing values per column
print("\n--- MISSING VALUE COUNT PER COLUMN ---")
print(df.isna().sum())

# Step 3: Remove duplicate records
df_dedup = df.drop_duplicates()
print(f"\n--- AFTER DEDUPLICATION (Rows removed: {len(df) - len(df_dedup)}) ---")
print(df_dedup)

# Step 4: Impute missing numerical values (Age with median, Income with mean)
df_clean = df_dedup.copy()
median_age = df_clean['age'].median()
mean_income = df_clean['monthly_income'].mean()

df_clean['age'] = df_clean['age'].fillna(median_age)
df_clean['monthly_income'] = df_clean['monthly_income'].fillna(mean_income)

# Step 5: Rename columns to clear professional names
df_clean = df_clean.rename(columns={
    'cust_id': 'Customer_ID',
    'cust_name': 'Full_Name',
    'age': 'Age',
    'monthly_income': 'Monthly_Income_INR'
})

print("\n--- FINAL CLEANED & RENAMED DATASET ---")
print(df_clean)
print("\nRemaining missing values:", df_clean.isna().sum().sum())


: 

In [ ]:
#3.
"""
DS-505 Practical: Memory Optimization using Dtype Downcasting
Simple student-friendly program based on the uploaded workbook.
"""

import pandas as pd
import numpy as np

# Step 1: Synthesize a 100,000-row dataset with unoptimized default types
np.random.seed(42)
n_rows = 100_000

df = pd.DataFrame({
    'User_ID': np.random.randint(1000, 9999, size=n_rows, dtype='int64'),
    'Age': np.random.randint(18, 70, size=n_rows, dtype='int64'),
    'Account_Balance': np.random.uniform(500.0, 50000.0, size=n_rows),
    'Subscription_Type': np.random.choice(['Basic', 'Standard', 'Premium', 'VIP'], size=n_rows),
    'Active_Status': np.random.choice(['Active', 'Inactive'], size=n_rows)
})

# Step 2: Measure baseline deep memory footprint
baseline_mem = df.memory_usage(deep=True).sum() / (1024 * 1024)
print("--- BASELINE MEMORY PROFILING ---")
print(df.dtypes)
print(f"Initial Memory Usage: {baseline_mem:.2f} MB")

# Step 3: Perform intelligent dtype optimization
df_opt = df.copy()

# Downcast integers
df_opt['User_ID'] = pd.to_numeric(df_opt['User_ID'], downcast='integer')
df_opt['Age'] = pd.to_numeric(df_opt['Age'], downcast='integer')

# Downcast floats
df_opt['Account_Balance'] = pd.to_numeric(df_opt['Account_Balance'], downcast='float')

# Convert repetitive object strings to category
df_opt['Subscription_Type'] = df_opt['Subscription_Type'].astype('category')
df_opt['Active_Status'] = df_opt['Active_Status'].astype('category')

# Step 4: Measure optimized memory usage
opt_mem = df_opt.memory_usage(deep=True).sum() / (1024 * 1024)
savings_pct = ((baseline_mem - opt_mem) / baseline_mem) * 100

print("\n--- OPTIMIZED MEMORY PROFILING ---")
print(df_opt.dtypes)
print(f"Optimized Memory Usage: {opt_mem:.2f} MB")
print(f"Memory Footprint Reduction: {savings_pct:.2f}% RAM Saved!")


In [ ]:
#4.
"""
DS-505 Practical: Batch CSV Processing using Chunking
Simple student-friendly program based on the uploaded workbook.
"""

import pandas as pd
import numpy as np

# Step 1: Generate a sample CSV file on disk with 5,000 transaction records
csv_filename = "transactions_stream.csv"
records = 5000

df_temp = pd.DataFrame({
    'Txn_ID': range(1, records + 1),
    'Amount': np.random.uniform(10.0, 500.0, size=records).round(2),
    'Category': np.random.choice(['Retail', 'Dining', 'Travel', 'Utilities'], size=records)
})
df_temp.to_csv(csv_filename, index=False)
print(f"Created '{csv_filename}' with {records:,} records.")

# Step 2: Process the CSV file in chunks to avoid high RAM consumption
chunk_size = 1000
total_records = 0
running_sum = 0.0
chunk_number = 1

print(f"\n--- PROCESSING IN CHUNKS OF {chunk_size} ROWS ---")
for chunk in pd.read_csv(csv_filename, chunksize=chunk_size):
    chunk_count = len(chunk)
    chunk_sum = chunk['Amount'].sum()
    
    total_records += chunk_count
    running_sum += chunk_sum
    
    print(f"Chunk {chunk_number}: Processed {chunk_count} rows | Chunk Sum: ${chunk_sum:,.2f} | Running Total: ${running_sum:,.2f}")
    chunk_number += 1

overall_average = running_sum / total_records
print("\n--- FINAL AGGREGATION METRICS ---")
print(f"Total Records Processed : {total_records:,}")
print(f"Total Cumulative Volume : ${running_sum:,.2f}")
print(f"Overall Average Value   : ${overall_average:.2f}")


In [ ]:
#5.
"""
DS-505 Practical: PySpark Session and DataFrame Schema Inspection
Simple student-friendly program based on the uploaded workbook.
"""

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Step 1: Initialize PySpark SparkSession in local cluster mode
spark = SparkSession.builder \
    .appName("DS505_Unit2_Practical5") \
    .master("local[*]") \
    .getOrCreate()

print("--- SPARK SESSION DETAILS ---")
print(f"Spark Application Name : {spark.sparkContext.appName}")
print(f"Spark Version          : {spark.version}")
print(f"Spark Master Mode      : {spark.sparkContext.master}")

# Step 2: Define structured employee data and schema
data = [
    ("E101", "Aarav Sharma", "Engineering", 75000.0, "Surat"),
    ("E102", "Diya Patel", "Analytics", 68000.0, "Ahmedabad"),
    ("E103", "Rohan Mehta", "Engineering", 82000.0, "Surat"),
    ("E104", "Sneha Joshi", "Marketing", 54000.0, "Vadodara"),
    ("E105", "Kabir Shah", "Analytics", 71000.0, "Surat")
]

schema = StructType([
    StructField("Emp_ID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Salary", DoubleType(), True),
    StructField("City", StringType(), True)
])

# Step 3: Create DataFrame and inspect structure
df = spark.createDataFrame(data, schema=schema)

print("\n--- PYSPARK DATAFRAME SCHEMA HIERARCHY ---")
df.printSchema()

print("\n--- PYSPARK DATAFRAME RECORDS (.show) ---")
df.show()


In [ ]:
#6.
"""
DS-505 Practical: PySpark Selection, Filtering and Column Transformations
Simple student-friendly program based on the uploaded workbook.
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DS505_Transformations").master("local[*]").getOrCreate()

# Create sample dataset
data = [
    (1, "Aarav", "IT", 65000, "Surat"),
    (2, "Diya", "HR", 45000, "Navsari"),
    (3, "Rohan", "IT", 72000, "Surat"),
    (4, "Pooja", "Finance", 58000, "Ahmedabad"),
    (5, "Kabir", "IT", 81000, "Surat"),
    (6, "Neha", "Finance", 62000, "Surat")
]
columns = ["ID", "Name", "Dept", "Salary", "City"]
df = spark.createDataFrame(data, columns)

print("--- ORIGINAL DATAFRAME ---")
df.show()

# 1. Column Selection
print("--- 1. SELECT NAME AND SALARY ---")
df.select("Name", "Salary").show()

# 2. Multi-condition Filtering
print("--- 2. FILTER (Dept == 'IT' AND Salary > 70000) ---")
df.filter((col("Dept") == "IT") & (col("Salary") > 70000)).show()

# 3. Add Computed Column (Bonus = 10% of Salary)
print("--- 3. ADD COMPUTED 'Bonus' COLUMN ---")
df_with_bonus = df.withColumn("Bonus", col("Salary") * 0.10)
df_with_bonus.show()

# 4. Rename Column
print("--- 4. RENAME 'Salary' TO 'Base_Salary' ---")
df_renamed = df_with_bonus.withColumnRenamed("Salary", "Base_Salary")
df_renamed.show()


In [ ]:
#7.
"""
DS-505 Practical: PySpark Grouped Aggregations
Simple student-friendly program based on the uploaded workbook.
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, max, sum, round

spark = SparkSession.builder.appName("DS505_Aggregations").master("local[*]").getOrCreate()

# Create department dataset
data = [
    ("Amit", "Sales", 45000),
    ("Pooja", "Engineering", 85000),
    ("Rahul", "Sales", 52000),
    ("Sneha", "Marketing", 48000),
    ("Vikram", "Engineering", 92000),
    ("Ananya", "Marketing", 54000),
    ("Karan", "Sales", 49000),
    ("Riya", "Engineering", 88000)
]
df = spark.createDataFrame(data, ["Employee", "Department", "Salary"])

print("--- INPUT DATASET ---")
df.show()

# GroupBy Aggregation
print("--- DEPARTMENT-WISE AGGREGATION METRICS ---")
dept_summary = df.groupBy("Department").agg(
    count("*").alias("Headcount"),
    round(avg("Salary"), 2).alias("Average_Salary"),
    max("Salary").alias("Max_Salary"),
    sum("Salary").alias("Total_Payroll")
).orderBy(col("Average_Salary").desc())

dept_summary.show()

# Action execution
print(f"Total Unique Departments: {dept_summary.count()}")
print(f"Total Records in Dataset : {df.count()}")


In [ ]:
#8.
"""
DS-505 Practical: Parquet Storage using PySpark
Simple student-friendly program based on the uploaded workbook.
"""

from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.appName("DS505_Parquet").master("local[*]").getOrCreate()

# Step 1: Create sample IoT sensor telemetry DataFrame
data = [
    ("Sensor_01", "Temperature", 24.5, "2026-10-10 10:00:00"),
    ("Sensor_02", "Humidity", 65.2, "2026-10-10 10:00:00"),
    ("Sensor_01", "Temperature", 25.1, "2026-10-10 10:05:00"),
    ("Sensor_03", "Pressure", 1013.25, "2026-10-10 10:05:00"),
    ("Sensor_02", "Humidity", 64.8, "2026-10-10 10:10:00")
]
columns = ["Sensor_ID", "Metric_Type", "Metric_Value", "Timestamp"]
df = spark.createDataFrame(data, columns)

print("--- ORIGINAL SENSOR DATAFRAME ---")
df.show()

# Step 2: Write to Apache Parquet columnar file format
parquet_path = "sensor_telemetry.parquet"
df.write.mode("overwrite").parquet(parquet_path)
print(f"Successfully serialized DataFrame to Parquet storage at: '{parquet_path}'")

# Step 3: Read back from Parquet format
df_parquet = spark.read.parquet(parquet_path)

print("\n--- RELOADED FROM PARQUET FORMAT ---")
df_parquet.printSchema()
df_parquet.show()

print(f"Verified Record Count: {df_parquet.count()} rows.")


In [ ]:
#9.
"""
DS-505 Practical: Missing Value Imputation and Categorical Encoding
Simple student-friendly program based on the uploaded workbook.
"""

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# Step 1: Create raw dataset with missing values and categorical features
raw_data = {
    'Applicant_ID': [101, 102, 103, 104, 105, 106],
    'Credit_Score': [710, np.nan, 680, 750, np.nan, 620],
    'Income_k': [55.0, 62.0, np.nan, 80.0, 48.0, 50.0],
    'Education': ['Bachelors', 'Masters', 'PhD', 'Bachelors', 'Masters', 'Bachelors'],
    'Loan_Approved': ['Yes', 'No', 'Yes', 'Yes', 'No', 'No']
}
df = pd.DataFrame(raw_data)
print("--- ORIGINAL DATASET ---")
print(df)

# Step 2: Handle Missing Numerical Values with SimpleImputer
imputer = SimpleImputer(strategy='mean')
df[['Credit_Score', 'Income_k']] = imputer.fit_transform(df[['Credit_Score', 'Income_k']])

# Step 3: Encode Target Variable using LabelEncoder
le = LabelEncoder()
df['Loan_Approved_Encoded'] = le.fit_transform(df['Loan_Approved'])

# Step 4: One-Hot Encode 'Education' category
df_encoded = pd.get_dummies(df, columns=['Education'], prefix='Edu', drop_first=False)

print("\n--- PROCESSED & ENCODED MACHINE LEARNING READY DATASET ---")
print(df_encoded.drop(columns=['Loan_Approved']))
print(f"\nLabelEncoder Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")


In [ ]:
#10.
"""
DS-505 Practical: Feature Scaling and Train-Test Split
Simple student-friendly program based on the uploaded workbook.
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Step 1: Create feature matrix X and target y
np.random.seed(42)
X = pd.DataFrame({
    'Feature_Age': [20, 25, 30, 35, 40, 45, 50, 55, 60, 65],
    'Feature_Salary': [25000, 32000, 48000, 55000, 62000, 71000, 85000, 92000, 110000, 130000]
})
y = np.array([0, 0, 0, 1, 0, 1, 1, 1, 1, 1])

print("--- ORIGINAL FEATURE VALUES ---")
print(X)

# Step 2: Apply StandardScaler (Mean=0, Std=1)
std_scaler = StandardScaler()
X_standardized = std_scaler.fit_transform(X)

# Step 3: Apply MinMaxScaler (Range [0, 1])
minmax_scaler = MinMaxScaler()
X_normalized = minmax_scaler.fit_transform(X)

print("\n--- STANDARDIZED FEATURES (StandardScaler: mu=0, sigma=1) ---")
print(pd.DataFrame(X_standardized, columns=X.columns).round(3).head(4))

print("\n--- NORMALIZED FEATURES (MinMaxScaler: [0, 1]) ---")
print(pd.DataFrame(X_normalized, columns=X.columns).round(3).head(4))

# Step 4: Partition dataset into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_standardized, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\n--- PARTITION SIZES ---")
print(f"Training Features Shape : {X_train.shape} | Training Labels: {len(y_train)}")
print(f"Testing Features Shape  : {X_test.shape}  | Testing Labels : {len(y_test)}")


In [ ]:
#11.
"""
DS-505 Practical: Linear Regression for Salary Prediction
Simple student-friendly program based on the uploaded workbook.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Step 1: Create Experience vs Salary dataset
np.random.seed(42)
experience = np.array([1.1, 1.5, 2.0, 2.9, 3.2, 3.7, 4.0, 4.5, 5.1, 5.9, 6.8, 7.1, 7.9, 8.7, 9.5, 10.3]).reshape(-1, 1)
salary = np.array([39343, 46205, 37731, 56642, 54445, 57189, 63218, 61111, 66029, 81363, 91738, 98273, 101302, 113812, 109431, 122391])

# Step 2: Split into Train (75%) and Test (25%)
X_train, X_test, y_train, y_test = train_test_split(experience, salary, test_size=0.25, random_state=42)

# Step 3: Instantiate and train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 4: Extract model parameters
slope = model.coef_[0]
intercept = model.intercept_
print("--- TRAINED MODEL EQUATION ---")
print(f"Salary = ({slope:.2f} * Experience) + {intercept:.2f}")

# Step 5: Predict on Test Set and Evaluate
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- TEST EVALUATION METRICS ---")
comparison = pd.DataFrame({'Years_Exp': X_test.flatten(), 'Actual_Salary': y_test, 'Predicted_Salary': y_pred.round(2)})
print(comparison)
print(f"\nMean Squared Error (MSE) : {mse:,.2f}")
print(f"R-squared Score (R2)     : {r2:.4f} (High predictive accuracy)")


In [ ]:
#12.
"""
DS-505 Practical: Decision Tree and KNN Classification
Simple student-friendly program based on the uploaded workbook.
"""

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Step 1: Load standard Iris multiclass dataset
iris = load_iris()
X, y = iris.data, iris.target

# Step 2: Split into Train (70%) and Test (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Step 3: Train Decision Tree Classifier
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
dt_acc = accuracy_score(y_test, dt_preds)

# Step 4: Train K-Nearest Neighbors Classifier (K=5)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
knn_preds = knn_model.predict(X_test)
knn_acc = accuracy_score(y_test, knn_preds)

# Step 5: Compare Performance
print("--- MODEL EVALUATION & COMPARISON ---")
print(f"1. Decision Tree Accuracy : {dt_acc * 100:.2f}%")
print("Confusion Matrix (Decision Tree):")
print(confusion_matrix(y_test, dt_preds))

print(f"\n2. K-Nearest Neighbors Accuracy (K=5) : {knn_acc * 100:.2f}%")
print("Confusion Matrix (KNN):")
print(confusion_matrix(y_test, knn_preds))


In [ ]:
#13.
"""
DS-505 Practical: Text Cleaning and Tokenization
Simple student-friendly program based on the uploaded workbook.
"""

import string

# Step 1: Raw noisy text string
raw_text = "   *** Big Data Handling & Machine Learning with Apache Spark, PySpark & LLMs! Are you Ready???   "

print("--- RAW TEXT INPUT ---")
print(f"'{raw_text}'")

# Step 2: Text Normalization (Lowercasing & Whitespace Stripping)
cleaned_text = raw_text.strip().lower()

# Step 3: Punctuation Removal using replace / string translation
punctuation_marks = string.punctuation
for char in punctuation_marks:
    cleaned_text = cleaned_text.replace(char, " ")

# Step 4: Normalize multiple spaces to single space
cleaned_text = " ".join(cleaned_text.split())

print("\n--- CLEANED & NORMALIZED TEXT ---")
print(f"'{cleaned_text}'")

# Step 5: Word Tokenization using split()
tokens = cleaned_text.split()

print("\n--- EXTRACTED WORD TOKENS ---")
print(f"Token Count : {len(tokens)}")
print(f"Tokens List : {tokens}")


In [ ]:
#14.
"""
DS-505 Practical: Vocabulary Frequency and Stopword Filtering
Simple student-friendly program based on the uploaded workbook.
"""

from collections import Counter

# Step 1: Text Corpus
corpus = """
Big data refers to massive volumes of structured and unstructured data.
Apache Spark is a distributed computing system designed for big data processing.
Machine learning models learn patterns from large scale big data.
Natural language processing and transformers enable intelligent understanding of text data.
"""

# Step 2: Define basic English stop words list
stopwords = {'to', 'of', 'and', 'a', 'is', 'for', 'from', 'in'}

# Step 3: Clean and tokenize corpus
words = corpus.lower().replace('.', '').replace(',', '').split()

# Step 4: Filter out stop words
filtered_words = [w for w in words if w not in stopwords]

# Step 5: Count word frequencies
frequency_counter = Counter(filtered_words)

print("--- VOCABULARY FREQUENCY ANALYSIS ---")
print(f"Total Raw Words         : {len(words)}")
print(f"After Stopword Removal  : {len(filtered_words)}")
print(f"Total Unique Vocabulary : {len(frequency_counter)}")

print("\n--- TOP 5 MOST FREQUENT KEYWORDS ---")
for word, freq in frequency_counter.most_common(5):
    print(f"Keyword: {word:<15} | Frequency: {freq}")


In [ ]:
#15.
"""
DS-505 Practical: Sentiment Analysis using Hugging Face
Simple student-friendly program based on the uploaded workbook.
"""

from transformers import pipeline

# Step 1: Initialize pretrained Hugging Face Sentiment Analysis Pipeline
print("Loading Hugging Face sentiment-analysis pipeline...")
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Step 2: Define sample product & service customer reviews
reviews = [
    "PySpark makes distributed computing on massive datasets incredibly fast and enjoyable!",
    "The dataset had too many corrupted records and missing values, causing the training to crash.",
    "Transformers and Large Language Models are revolutionizing artificial intelligence.",
    "The data engineering pipeline was slow and poorly documented."
]

# Step 3: Run inference and display results
print("\n--- PREDICTED SENTIMENT SCORES ---")
for i, review in enumerate(reviews, 1):
    result = classifier(review)[0]
    label = result['label']
    score = result['score'] * 100
    print(f"Review {i}: '{review}'")
    print(f"Result   : [{label}] with {score:.2f}% confidence\n")


In [ ]:
#16.
"""
DS-505 Practical: Text Summarization using Hugging Face
Simple student-friendly program based on the uploaded workbook.
"""

from transformers import pipeline

# Step 1: Initialize pretrained Hugging Face Summarization Pipeline
print("Loading Hugging Face summarization pipeline...")
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

# Step 2: Define an extended article on Big Data & AI
article = """
Big Data has fundamentally changed the landscape of artificial intelligence and machine learning.
Traditionally, organizations relied on relational databases that scaled vertically by adding more compute
resources to a single server. However, with the exponential growth of web traffic, social media feeds,
and IoT sensors, modern data platforms transitioned to distributed frameworks like Apache Spark.
Apache Spark utilizes in-memory computing and Directed Acyclic Graphs to process petabytes of information
across commodity hardware clusters. Furthermore, the advent of Self-Attention Transformers and Large Language
Models has created an insatiable demand for high-quality, web-scale corpora, solidifying big data engineering
as the core foundational backbone of modern Generative AI.
"""

# Step 3: Generate summary with controlled length
summary = summarizer(article, max_length=60, min_length=25, do_sample=False)

print("\n--- ORIGINAL ARTICLE LENGTH ---")
print(f"Word Count: {len(article.split())} words")

print("\n--- GENERATED CONDENSED SUMMARY ---")
print(summary[0]['summary_text'])
print(f"Summary Word Count: {len(summary[0]['summary_text'].split())} words")
